# Descriptive Statistics — Block 2, Week 2
**Aayan Mulani │ Decimal Point Analytics Preparation**

Descriptive statistics are the first thing you compute before any modelling.
They tell you the shape, centre, and spread of your data.
In finance, these numbers reveal how returns behave — are they symmetric?
Are crashes more common than rallies? How fat are the tails?

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

## 1. Downloading Data
We download Nifty 50 daily closing prices from 2020 to 2025.
We then compute daily percentage returns — this is what we run statistics on.
Raw prices are not stationary (they trend upward), but returns are.
We multiply by 100 to express returns as percentages (e.g. 0.85% instead of 0.0085).

In [ ]:
nifty = yf.download('^NSEI', start = '2020-01-01', end = '2025-01-01')['Close']
returns = nifty.pct_change().dropna()*100

print(f'Total Trading Days: {len(returns)}')
print(f'Date Range : {returns.index[0].date()} to {returns.index[-1].date()}')
print(f'\nFirst 5 Returns:')
print(returns.head())

## 2. Core Descriptive Statistics
We compute 8 key statistics on Nifty daily returns.
Each number tells us something different about the shape and behaviour of returns.

In [ ]:
print(f'Mean:     {returns.mean().values[0]:.4f}%')
print(f'Median:   {returns.median().values[0]:.4f}%')
print(f'Std Dev:  {returns.std().values[0]:.4f}%')
print(f'Variance: {returns.var().values[0]:.4f}')
print(f'Skewness: {returns.skew().values[0]:.4f}')
print(f'Kurtosis: {returns.kurtosis().values[0]:.4f}')
print(f'Min:      {returns.min().values[0]:.4f}%')
print(f'Max:      {returns.max().values[0]:.4f}%')

## 3. Interpretation

**Mean vs Median:**
Mean is lower than median because there are few very extreme negative days (like the COVID crash) pulling the mean down. The median is unaffected because it only cares about the middle value, not the size of extremes.

**Skewness (-1.39):**
The negative sign shows that data is negatively skewed, that means most of the data points are on right sides but on left sides there are huge negative points making it left tailed data. The main reason for this skewness is covid pandemic

**Kurtosis (18.97):**
The Kurtosis value of 18.97 is extremely higher than the normal one (3). It means that tails are drastically fatter than normal tails and probability of extreme events very higher than a normal distribution can predict.

**Key takeaway:**
From this we can infer that market is highly volatile with extreme events and standard models that assume normality will underestimate the probability of extreme losses.

---
## Section 2: Probability Distributions & Normality Testing

In Section 1, we described *what* Nifty 50 returns look like using summary statistics.
In this section, we go deeper — we ask *what shape* the distribution of returns follows,
whether it resembles a normal distribution, and what that means for financial modelling.

**Concepts covered:**
- Normal distribution and the 68-95-99.7 rule
- Z-scores
- Confidence Intervals
- Jarque-Bera Normality Test

In [ ]:
# --- Imports ---
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm, jarque_bera

# --- Reload Nifty 50 Data ---
nifty = yf.download("^NSEI", start="2020-01-01", end="2025-01-01", auto_adjust=True)
returns = nifty["Close"].squeeze().pct_change().dropna()

print(f"Data loaded: {len(returns)} trading days")
print(f"Mean return : {returns.mean():.4f}")
print(f"Std dev     : {returns.std():.4f}")

In [ ]:
# --- Histogram vs Normal Distribution ---

mean = returns.mean()
std  = returns.std()

# Generate x values for the normal curve
x = np.linspace(returns.min(), returns.max(), 200)

# Compute the normal curve y values
normal_curve = norm.pdf(x, mean, std)

# Plot
fig, ax = plt.subplots(figsize=(10, 5))

ax.hist(returns, bins=80, density=True, color="#85B7EB",
        edgecolor="white", alpha=0.7, label="Actual Nifty Returns")

ax.plot(x, normal_curve, color="#D04A2A", linewidth=2.5,
        label="Normal Distribution (fitted)")

ax.axvline(mean, color="#185FA5", linestyle="--",
           linewidth=1.5, label=f"Mean: {mean:.4f}")

ax.set_title("Nifty 50 Daily Returns vs Normal Distribution (2020–2025)",
             fontsize=14, fontweight="bold")
ax.set_xlabel("Daily Return", fontsize=12)
ax.set_ylabel("Density", fontsize=12)
ax.legend(fontsize=11)

plt.tight_layout()
plt.show()

### Interpretation: Actual Returns vs Normal Distribution

The histogram reveals three key deviations from normality:

1. **Taller, sharper peak (Leptokurtosis):** The actual return distribution is far more
   peaked at the centre than the normal curve predicts. Most days, Nifty moves very
   little — tighter clustering than the normal assumption allows.

2. **Fat tails:** The histogram shows isolated bars far into the left tail (around −0.10
   to −0.12), representing extreme crash days like March 2020. The normal distribution
   assigns near-zero probability to these events — dangerously underestimating real risk.

3. **Slight negative skew:** The left tail is visibly longer than the right tail,
   confirming that large negative days (crashes) are more extreme than large positive days.

These deviations mean models built on the normality assumption — such as basic
Value-at-Risk (VaR) — will systematically underestimate the probability of market crashes.

In [ ]:
# --- Z-Score Calculation ---

mean = returns.mean()
std  = returns.std()

# Calculate Z-score for every trading day
z_scores = (returns - mean) / std

# Most extreme crash days
print("=== Top 5 Crash Days (Most Negative Z-Scores) ===")
print(z_scores.nsmallest(5).round(2))

print()

# Most extreme rally days
print("=== Top 5 Rally Days (Most Positive Z-Scores) ===")
print(z_scores.nlargest(5).round(2))

In [ ]:
# --- Confidence Interval ---

mean = returns.mean()
std  = returns.std()
n    = len(returns)

# Standard error
se = std / np.sqrt(n)

# Z value for 95% confidence (1.96)
z_critical = norm.ppf(0.975)

# Confidence interval bounds
lower = mean - z_critical * se
upper = mean + z_critical * se

print(f"Sample Mean         : {mean:.6f}")
print(f"Standard Error      : {se:.6f}")
print(f"Z Critical (95%)    : {z_critical:.4f}")
print()
print(f"95% Confidence Interval: [{lower:.6f},  {upper:.6f}]")
print()
print(f"In plain English: We are 95% confident the true mean")
print(f"daily return of Nifty 50 lies between {lower*100:.4f}% and {upper*100:.4f}%")

In [ ]:
# --- Jarque-Bera Normality Test ---

skew     = returns.skew()
kurt     = returns.kurt()  # pandas returns excess kurtosis (normal = 0, not 3)
jb_stat, p_value = jarque_bera(returns)

print(f"Skewness              : {skew:.4f}")
print(f"Excess Kurtosis       : {kurt:.4f}")
print()
print(f"Jarque-Bera Statistic : {jb_stat:.4f}")
print(f"P-Value               : {p_value:.6f}")
print()

decision = "Reject" if p_value < 0.05 else "Fail to Reject"
print(f"Decision (at 5% significance): {decision} normality")

### Jarque-Bera Test Results & Final Interpretation

| Metric               | Value     | Normal Distribution Benchmark |
|----------------------|-----------|-------------------------------|
| Skewness             | -1.3945   | 0 (perfectly symmetric)       |
| Excess Kurtosis      | 18.9748   | 0 (normal = 3, excess = 0)    |
| JB Statistic         | 18782.89  | ~0                            |
| P-Value              | ~0.000000 | > 0.05 to accept normality    |
| Decision             | **Reject normality** | —                |

**What this means for financial modelling:**

1. **Negative skew (−1.39):** Nifty returns are not symmetric. Extreme negative days
   (crashes) are significantly more severe than extreme positive days. Any model
   assuming symmetry will underestimate downside risk.

2. **Excess kurtosis (18.97):** The distribution has an extremely sharp peak and fat
   tails — a pattern called leptokurtosis. The March 2020 crash (Z = −10.83) is a
   prime example of a fat-tail event that a normal model would assign near-zero
   probability to.

3. **Practical implication:** Models like basic Value-at-Risk (VaR) that assume
   normality will systematically underestimate the probability of crashes. This
   is why quants use fat-tailed distributions (Student's t, Extreme Value Theory)
   for serious risk modelling.

**Conclusion:** Nifty 50 daily returns are definitively non-normal — a finding
consistent with virtually all equity markets globally.

---
## Section 3: Hypothesis Testing — Are Nifty Returns Significantly Positive?

Hypothesis testing lets us determine whether a pattern in data is statistically significant 
or could simply be due to random chance.

**Null Hypothesis (H0):** Nifty's mean daily return = 0 (no real positive drift)  
**Alternative Hypothesis (H1):** Nifty's mean daily return ≠ 0  

We use a one-sample t-test to decide whether to reject H0.

In [ ]:
from scipy.stats import ttest_1samp

# One-sample t-test: H0 = mean daily return is 0
t_stat, p_value = ttest_1samp(returns.dropna(), popmean=0)

print(f'T-statistic: {t_stat:.4f}')
print(f'P-value:     {p_value:.6f}')

if p_value < 0.05:
    print('Reject H0: Nifty returns are significantly different from zero')
else:
    print('Fail to reject H0: no significant evidence returns differ from zero')

### Interpretation — One-Sample t-Test

- **T-statistic: 1.781** — the sample mean is 1.781 standard errors above zero
- **P-value: 0.075** — 7.5% probability this result is due to random chance
- **Decision: Fail to reject H0** — we cannot statistically confirm Nifty has a 
  positive daily return at the 95% confidence level

This does not mean returns are zero — it means our data does not provide strong 
enough evidence to prove otherwise. The COVID crash (2020) significantly increases 
volatility, making the signal harder to detect.

In [ ]:
from scipy.stats import ttest_ind

# Split returns into pre-COVID and post-COVID periods
pre_covid  = returns[returns.index < '2020-03-01']
post_covid = returns[returns.index >= '2020-06-01']

t2, p2 = ttest_ind(pre_covid.dropna(), post_covid.dropna())

print(f'Pre-COVID  mean: {pre_covid.mean():.4f}%')
print(f'Post-COVID mean: {post_covid.mean():.4f}%')
print(f'T-statistic: {t2:.4f}')
print(f'P-value:     {p2:.4f}')

if p2 < 0.05:
    print('Reject H0: returns before and after COVID are significantly different')
else:
    print('Fail to reject H0: no significant difference between the two periods')

### Interpretation — Two-Sample t-Test (Pre vs Post COVID)

- **Pre-COVID mean: -0.0020%** — slightly negative, dragged down by the Feb-Mar 2020 crash
- **Post-COVID mean: +0.0008%** — positive, reflecting the strong recovery rally post-June 2020
- **P-value: 0.0571** — just above the 0.05 threshold; borderline significant

**Decision: Fail to reject H0** — but only barely. A p-value of 0.057 suggests the 
two periods behave differently, but we cannot confirm this at the 95% confidence level.

In practice, an analyst would treat this as a meaningful structural shift worth 
monitoring, rather than dismissing it entirely.